# Prepare Routing Inputs

Create the active 100 m routing-cell table used by the Valhalla workflows. This notebook consumes the existing analytical foundation and does not rebuild the raster.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

def find_project_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "README.md").exists() and (path / "ANAL").exists() and (path / "TOOLS").exists():
            return path
    raise FileNotFoundError("Could not find project root")


PROJECT_DIR = find_project_dir(Path.cwd())
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
INPUT_DIR = ROUTING_DATA / "inputs"

RASTER_PATH = ANAL_DATA / "raster_100m_styria.geoparquet"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
FIRMS_PATH = ANAL_DATA / "firms_assigned_100m.geoparquet"
OUTPUT_PATH = INPUT_DIR / "active_routing_cells_100m.parquet"

CRS_ANALYSIS = "EPSG:3035"
CRS_ROUTING = "EPSG:4326"

INPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
raster = gpd.read_parquet(RASTER_PATH).to_crs(CRS_ANALYSIS)
panel = pd.read_parquet(PANEL_PATH, columns=["grid_id"])
firms = gpd.read_parquet(FIRMS_PATH)

active_grid_ids = pd.Index(panel["grid_id"].dropna().unique(), name="grid_id")
firm_grid_ids = pd.Index(firms["grid_id_100m"].dropna().unique(), name="grid_id")

active_cells = raster[raster["grid_id"].isin(active_grid_ids)].copy()
active_cells["active_panel_cell"] = True
active_cells["has_2025_population"] = active_cells["population_2025"].fillna(0) > 0
active_cells["has_firm_observation"] = active_cells["grid_id"].isin(firm_grid_ids)

centroids_3035 = active_cells.geometry.centroid
active_cells["centroid_x_3035"] = centroids_3035.x
active_cells["centroid_y_3035"] = centroids_3035.y

routing_points = active_cells.set_geometry(centroids_3035).to_crs(CRS_ROUTING)
active_cells["lon"] = routing_points.geometry.x
active_cells["lat"] = routing_points.geometry.y

columns = [
    "grid_id",
    "municipality_id",
    "municipality_name",
    "population_2025",
    "centroid_x_3035",
    "centroid_y_3035",
    "lon",
    "lat",
    "active_panel_cell",
    "has_2025_population",
    "has_firm_observation",
]

active_cells[columns].to_parquet(OUTPUT_PATH, index=False)
print(f"Wrote {len(active_cells):,} active routing cells to {OUTPUT_PATH}")

In [ ]:
check = active_cells[columns]
assert check["grid_id"].is_unique
assert check["lon"].between(9, 18).all()
assert check["lat"].between(45, 50).all()
check.head()